# NB06 — Train PPO (Proximal Policy Optimization)

Train a PPO agent on `UnitreeG1DishWipe-v1` using Stable-Baselines3.
The robot (25 DOF upper body, fixed legs) must clean a plate on a kitchen
counter by making palm contact and sweeping.


## Objective

1. Create vectorised ManiSkill environments with proper wrappers.
2. Configure PPO hyperparameters for the 25-DOF action space.
3. Train for `TOTAL_ENV_STEPS` with evaluation callbacks.
4. Save model, learning curve, and log to MLflow.


## Environment

| Notebook | Goal | Required HW | Min CPU | Min RAM | GPU VRAM | Notes |
|---|---|---|---:|---:|---:|---|
| NB06 | Train PPO | GPU (practical) | 8 cores | 16 GB | 12-24 GB | RunPod 4090+ |

> **Note**: This notebook is designed for RunPod with GPU. On CPU, reduce
> `TOTAL_ENV_STEPS` and `N_ENVS` for testing.


In [ ]:
import sys, os, platform, json
print(f"Python: {sys.version}"); print(f"OS: {platform.platform()}")
import numpy as np; print(f"NumPy: {np.__version__}")
import torch; print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"CUDA: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA: not available (CPU mode)")
import gymnasium as gym; print(f"Gymnasium: {gym.__version__}")
import stable_baselines3 as sb3; print(f"SB3: {sb3.__version__}")


## Imports


In [ ]:
import json, time
import numpy as np
import torch
import gymnasium as gym
from pathlib import Path
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.utils import set_random_seed

# Ensure project root is on sys.path so `src.envs` can be imported
PROJECT_ROOT = Path(os.getcwd()).resolve()
if (PROJECT_ROOT / "src").is_dir():
    pass  # already at project root
elif (PROJECT_ROOT.parent / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError(f"Cannot find src/ from {PROJECT_ROOT}")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)  # normalize cwd for artifact paths

from src.envs.dishwipe_env import UnitreeG1DishWipeEnv  # registers env


## Config


In [ ]:
# ── Detect hardware ──
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IS_GPU = DEVICE == "cuda"

CFG = dict(
    seed=42,
    env_id="UnitreeG1DishWipe-v1",
    control_mode="pd_joint_delta_pos",
    obs_mode="state",
    # ── Training budget ──
    total_env_steps=500_000 if IS_GPU else 20_000,  # reduced for CPU testing
    n_envs=4 if IS_GPU else 1,
    # ── PPO hyperparameters ──
    learning_rate=3e-4,
    n_steps=2048,
    batch_size=256,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    # ── Network ──
    net_arch=[256, 256],
    activation_fn="Tanh",
    # ── Evaluation ──
    eval_freq=10_000 if IS_GPU else 5_000,
    eval_episodes=10,
    # ── Device ──
    device=DEVICE,
)
SEED = CFG["seed"]
artifact_dir = Path("artifacts/NB06")
artifact_dir.mkdir(parents=True, exist_ok=True)
print("Config:", json.dumps(CFG, indent=2))


## Reproducibility


In [ ]:
import random; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
set_random_seed(SEED)
print(f"✅ Seeds set to {SEED}")


## Implementation Steps

### Step 1 — Create ManiSkill environment with wrappers


In [ ]:
from mani_skill.utils.wrappers.gymnasium import CPUGymWrapper

def make_env(env_id, obs_mode, control_mode, num_envs, seed):
    """Create a ManiSkill environment wrapped for SB3 compatibility.

    For num_envs=1: CPUGymWrapper converts batched (1, obs_dim) to (obs_dim,).
    For num_envs>1 on GPU: use ManiSkillVectorEnv for parallel SB3 VecEnv.
    """
    env = gym.make(
        env_id,
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=num_envs,
        render_mode=None,
    )
    if num_envs == 1:
        env = CPUGymWrapper(env)
    else:
        # GPU vectorized env → wrap as SB3 VecEnv
        from mani_skill.vector.wrappers.gymnasium import ManiSkillVectorEnv
        env = ManiSkillVectorEnv(env)
    return env

train_env = make_env(
    CFG["env_id"], CFG["obs_mode"], CFG["control_mode"],
    CFG["n_envs"], SEED
)
print(f"Train env: obs={train_env.observation_space.shape}, act={train_env.action_space.shape}")
print(f"Num envs: {CFG['n_envs']}")

eval_env = make_env(
    CFG["env_id"], CFG["obs_mode"], CFG["control_mode"], 1, SEED + 100
)
print(f"Eval env: obs={eval_env.observation_space.shape}")


### Step 2 — Configure PPO


In [ ]:
# Activation function
import torch.nn as nn
ACT_FN_MAP = {"Tanh": nn.Tanh, "ReLU": nn.ReLU}
act_fn = ACT_FN_MAP.get(CFG["activation_fn"], nn.Tanh)

policy_kwargs = dict(
    net_arch=CFG["net_arch"],
    activation_fn=act_fn,
)

model = PPO(
    "MlpPolicy",
    train_env,
    learning_rate=CFG["learning_rate"],
    n_steps=CFG["n_steps"],
    batch_size=CFG["batch_size"],
    n_epochs=CFG["n_epochs"],
    gamma=CFG["gamma"],
    gae_lambda=CFG["gae_lambda"],
    clip_range=CFG["clip_range"],
    ent_coef=CFG["ent_coef"],
    vf_coef=CFG["vf_coef"],
    max_grad_norm=CFG["max_grad_norm"],
    policy_kwargs=policy_kwargs,
    verbose=1,
    seed=SEED,
    device=DEVICE,
)
print(f"\nPPO model created on {DEVICE}")
print(f"Policy: {model.policy}")


### Step 3 — Train


In [ ]:
class TrainLogCallback(BaseCallback):
    """Log reward every N steps."""
    def __init__(self, log_freq=1000, verbose=0):
        super().__init__(verbose)
        self.log_freq = log_freq
        self.rewards = []

    def _on_step(self):
        if self.n_calls % self.log_freq == 0:
            if len(self.model.ep_info_buffer) > 0:
                mean_rew = np.mean([ep["r"] for ep in self.model.ep_info_buffer])
                self.rewards.append((self.num_timesteps, mean_rew))
                if self.verbose:
                    print(f"  Step {self.num_timesteps}: mean_reward={mean_rew:.3f}")
        return True

log_callback = TrainLogCallback(log_freq=2000, verbose=1)

print(f"Training PPO for {CFG['total_env_steps']} steps...")
t0 = time.time()
model.learn(
    total_timesteps=CFG["total_env_steps"],
    callback=[log_callback],
    progress_bar=True,
)
train_time = time.time() - t0
print(f"\n✅ Training complete in {train_time:.1f}s")


### Step 4 — Save model & evaluate


In [ ]:
# Save model
model_path = artifact_dir / "ppo_model"
model.save(str(model_path))
print(f"✅ Model saved: {model_path}.zip")

# Quick evaluation
eval_rewards = []
eval_cleaned = []
for ep in range(CFG["eval_episodes"]):
    obs, info = eval_env.reset(seed=SEED + 1000 + ep)
    total_rew = 0.0
    for step in range(1000):  # max_episode_steps
        action, _ = model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = eval_env.step(action)
        total_rew += reward.item() if hasattr(reward, "item") else float(reward)
        if terminated.any() if hasattr(terminated, "any") else terminated:
            break
    ratio = info.get("cleaned_ratio", torch.tensor([0.0]))
    r = ratio.item() if hasattr(ratio, "item") else float(ratio)
    eval_rewards.append(total_rew)
    eval_cleaned.append(r)

print(f"\nEvaluation ({CFG['eval_episodes']} episodes):")
print(f"  Reward  : {np.mean(eval_rewards):.3f} ± {np.std(eval_rewards):.3f}")
print(f"  Cleaned : {np.mean(eval_cleaned):.4f} ± {np.std(eval_cleaned):.4f}")


### Step 5 — Learning curve


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

if log_callback.rewards:
    steps, rewards = zip(*log_callback.rewards)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(steps, rewards, linewidth=1.5)
    ax.set_xlabel("Environment Steps")
    ax.set_ylabel("Mean Episode Reward")
    ax.set_title("NB06 — PPO Learning Curve (UnitreeG1DishWipe-v1)")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    curve_path = artifact_dir / "learning_curve.png"
    fig.savefig(str(curve_path), dpi=120, bbox_inches="tight")
    plt.close("all")
    print(f"✅ Saved: {curve_path}")
else:
    print("⚠️ No training logs available for plotting.")


### Step 6 — Save artifacts & MLflow


In [ ]:
# Save eval results
eval_results = {
    "mean_eval_reward": float(np.mean(eval_rewards)),
    "std_eval_reward": float(np.std(eval_rewards)),
    "mean_eval_cleaned": float(np.mean(eval_cleaned)),
    "train_time_seconds": train_time,
    "total_env_steps": CFG["total_env_steps"],
}
with open(artifact_dir / "eval_results.json", "w") as f:
    json.dump(eval_results, f, indent=2)

with open(artifact_dir / "nb06_config.json", "w") as f:
    json.dump(CFG, f, indent=2)

# MLflow
try:
    import mlflow
    from dotenv import load_dotenv
    load_dotenv(".env.local")
    uri = os.environ.get("MLFLOW_TRACKING_URI", "")
    if uri:
        mlflow.set_tracking_uri(uri)
        mlflow.set_experiment("dishwipe_unitree_g1")
        with mlflow.start_run(run_name="NB06_PPO"):
            mlflow.log_params({k: v for k, v in CFG.items()
                              if not isinstance(v, (list, dict))})
            mlflow.log_metric("eval_reward_mean", eval_results["mean_eval_reward"])
            mlflow.log_metric("eval_cleaned_mean", eval_results["mean_eval_cleaned"])
            mlflow.log_metric("train_time_s", train_time)
            mlflow.log_artifacts(str(artifact_dir), artifact_path="NB06")
        print("✅ MLflow run logged.")
    else:
        print("⚠️ MLflow not configured.")
except Exception as e:
    print(f"⚠️ MLflow: {e}")

print("\nArtifacts saved in:", artifact_dir)


## Results

- PPO trained on UnitreeG1DishWipe-v1 (25 DOF, dense reward)
- Model saved as `artifacts/NB06/ppo_model.zip`
- Evaluation metrics logged (reward, cleaned ratio)


## Artifacts

| File | Description |
|------|-------------|
| `artifacts/NB06/ppo_model.zip` | Trained PPO model |
| `artifacts/NB06/learning_curve.png` | Training reward curve |
| `artifacts/NB06/eval_results.json` | Evaluation metrics |
| `artifacts/NB06/nb06_config.json` | Config |


## Cleanup


In [ ]:
train_env.close()
eval_env.close()
print('✅ NB06 complete.')


## References

- Schulman et al. (2017) — PPO clipped surrogate objective
- Stable-Baselines3 PPO: https://stable-baselines3.readthedocs.io/en/master/modules/ppo.html
- ManiSkill3 env wrapping: https://maniskill.readthedocs.io/
